# admdongkor — demo & manual verification

이 노트북은 자동 pytest 가 못 잡는 부분(실제 네트워크 다운로드, 지도 시각, 한글 표시 등)을 **사람 눈으로** 확인하는 용도. 셀을 위→아래 순서로 실행.


## 0. 한글 폰트 설정 (matplotlib plot 용)


In [ ]:
import sys
import matplotlib
import matplotlib.pyplot as plt

if sys.platform == 'win32':
    matplotlib.rcParams['font.family'] = 'Malgun Gothic'
elif sys.platform == 'darwin':
    matplotlib.rcParams['font.family'] = 'AppleGothic'
else:
    # Linux: 'NanumGothic' 가 있으면 그걸로, 없으면 fallback
    from matplotlib import font_manager
    for cand in ['NanumGothic', 'Noto Sans CJK KR', 'UnDotum']:
        if any(cand in f.name for f in font_manager.fontManager.ttflist):
            matplotlib.rcParams['font.family'] = cand
            break

# 음수 기호(-)가 네모로 깨지는 것 방지
matplotlib.rcParams['axes.unicode_minus'] = False
print(f'matplotlib font: {matplotlib.rcParams["font.family"]}')


## 1. 버전 & 캐시 위치 확인


In [ ]:
import admdongkor as adk

print('version     :', adk.__version__)
print('cache dir   :', adk.cache_dir())
print('total keys  :', len(adk.get_list()))
print('first 3     :', adk.get_list()[:3])
print('last 3      :', adk.get_list()[-3:])
print('2025 keys   :', adk.get_list(year=2025))


## 2. `find()` — 행정구역명으로 버전 검색


In [ ]:
# 단일 토큰: 모든 레벨 substring
df = adk.find('종로')
print(f'rows: {len(df)}, versions: {df.version_key.nunique()}')
df.head(10)


In [ ]:
# 2 토큰: 자동으로 sgg 만 — 종로구 안의 읍면동은 줄줄이 안 나옴
adk.find('서울특별시 종로구').head()


In [ ]:
# 3 토큰: 자동으로 emd 만
adk.find('서울특별시 종로구 사직동').head()


In [ ]:
# 공백 무시 매칭: sggnm 이 '수원시권선구' 로 붙어 저장돼 있어도 매치
adk.find('수원시 권선구').head()


In [ ]:
# emd 행에는 code7 (통계청 7자리) 과 code8 이 같이 나옴
# 2023-10-01 통계청 7→8자리 전환 전후 비교
df = adk.find('사직동', year=[2023])
df[['version_key', 'code', 'code7', 'code8']]


In [ ]:
# 완전 일치 (단일 토큰만)
adk.find('종로구', exact=True).head()


In [ ]:
# level 명시로 자동 필터 override (종로구 내 모든 읍면동)
df = adk.find('서울특별시 종로구', level='emd')
print(f'rows: {len(df)}')
df.head()


In [ ]:
# 연도 범위
adk.find('세종', year=[2010, 2012])


## 3. `get()` — 지도 다운로드 & 첫 그림


In [ ]:
import time

t0 = time.time()
sido = adk.get('20250401', 'sido')
t_first = time.time() - t0

t0 = time.time()
sido2 = adk.get('20250401', 'sido')
t_cache = time.time() - t0

print(f'first download : {t_first:.2f}s')
print(f'cache hit      : {t_cache:.3f}s ({t_first/max(t_cache, 0.001):.0f}x faster)')
print(f'CRS            : EPSG:{sido.crs.to_epsg()}')
print(f'rows           : {len(sido)}')
sido.head()


In [ ]:
# 지도 그림 (한국 모양이 나와야 정상)
ax = sido.plot(figsize=(8, 10), edgecolor='black', linewidth=0.3)
ax.set_title('시도 — 20250401')
ax.set_axis_off()


## 4. 읍면동 지도 — 가장 큰 파일


In [ ]:
emd = adk.get('20250401', 'emd')
print(f'rows: {len(emd)}, cols: {list(emd.columns)}')
emd.head(3)


In [ ]:
# 서울만 뽑아 읍면동 경계 찍기 (전국 emd 3500개 plot 하면 버벅임)
seoul = emd[emd.sidocd == '11']
ax = seoul.plot(figsize=(10, 10), edgecolor='grey', linewidth=0.3, facecolor='none')
ax.set_title(f'서울 읍면동 — 20250401 ({len(seoul)} emds)')
ax.set_axis_off()


## 5. force_refresh 로 캐시 갱신


In [ ]:
f = adk.cache_dir() / 'sido_20250401.parquet'
before = f.stat().st_mtime
adk.get('20250401', 'sido', force_refresh=True)
after = f.stat().st_mtime
print(f'mtime before : {before}')
print(f'mtime after  : {after}')
print(f'updated      : {after > before}')


## 6. 캐시 폴더 현황


In [ ]:
files = sorted(adk.cache_dir().glob('*.parquet'))
total = sum(f.stat().st_size for f in files)
print(f'cache dir : {adk.cache_dir()}')
print(f'files     : {len(files)}')
print(f'total     : {total / 1024 / 1024:.1f} MB')
for f in files:
    print(f'  {f.stat().st_size / 1024 / 1024:6.2f} MB  {f.name}')


## 7. 시계열 — 같은 지역을 여러 해 비교


In [ ]:
# 세종시가 들어간 연도들의 sido 지도. 2012 이전엔 충청남도로 포함됨
import matplotlib.pyplot as plt

keys = ['20111231', '20121231', '20131231', '20181106']
fig, axes = plt.subplots(1, len(keys), figsize=(16, 6))
for ax, key in zip(axes, keys):
    g = adk.get(key, 'sido')
    g.plot(ax=ax, edgecolor='black', linewidth=0.3, facecolor='lightgrey')
    sejong = g[g.sidonm.astype(str).str.contains('세종', na=False)]
    if not sejong.empty:
        sejong.plot(ax=ax, facecolor='tomato', edgecolor='darkred')
    ax.set_title(key)
    ax.set_axis_off()
plt.suptitle('시도 경계 — 세종시 등장 시점 비교')
plt.tight_layout()
